# 🎮 Exploratory Data Analysis (EDA): video game catalog

**🎯 Goal:** understand the quality of the product data *before* building the duplicate detection pipeline.

With generative AI, it's tempting to throw raw data at a model with a prompt and hope for the best. 🙈 But that doesn't scale: every useless field is tokens you pay for on every call, and every inconsistent value is an error the model can spread. So before writing a single prompt, this EDA answers three questions:

1. 🧩 Which columns carry useful information, and which ones only add tokens?
2. 🧹 Which quality problems can be fixed with simple rules, no model needed?
3. 🔭 Which part of the catalog is in scope for the MVP?

**📦 Dataset:** [Amazon Reviews 2023](https://amazon-reviews-2023.github.io/), product metadata for the *Video Games* category (`meta_Video_Games.jsonl`), McAuley Lab, UCSD.

The notebook follows ten checks:

| # | Check | Question |
|---|---|---|
| 1 | 🗂️ Dataset structure | What does a record look like? |
| 2 | 🕳️ Data completeness | What is missing, including disguised nulls? |
| 3 | 🔑 Uniqueness and duplicates | Is there a reliable key? Are titles repeated? |
| 4 | 📊 Cardinality and value distribution | Are categories and labels consistent? |
| 5 | ✅ Format validity | Do values have the expected types? |
| 6 | 📈 Ranges and outliers | Are there impossible or extreme values? |
| 7 | 🔗 Cross-column consistency | Do related fields agree? |
| 8 | 📝 Text quality | How noisy are the titles? |
| 9 | 🪆 Nested-field analysis | What's inside the list and dict columns? |
| 10 | 🏁 Scope definition | Which products enter the pipeline? |

## ⚙️ Setup

Libraries and display options. Download `meta_Video_Games.jsonl` from the dataset page and save it in `data/`.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

DATA_PATH = "data/meta_Video_Games.jsonl"

## 🧰 Helper functions

Reusable functions for the checks below:

- `type_profile`: one row per column with its real Python types, nulls, empty values and distinct values. Pandas shows almost every column as `object`, which hides what's really inside.
- `clean_price`: turns the price into a number and flags "from X" prices.
- `plot_distribution`: histogram with density curve, median and 95th percentile.

In [ ]:
def type_profile(df: pd.DataFrame) -> pd.DataFrame:
    """Real Python types, nulls, empty values and distinct values per column."""
    rows = []
    for col in df.columns:
        types = df[col].map(lambda x: type(x).__name__).value_counts()
        rows.append({
            "column": col,
            "pandas_dtype": str(df[col].dtype),
            "real_types": dict(types),            # e.g. {'str': 62008, 'NoneType': 75261}
            "n_types": len(types),                # >1 = mixed column, review it
            "nulls": int(df[col].isna().sum()),
            "empty": int(df[col].map(
                lambda x: hasattr(x, "__len__") and not isinstance(x, str) and len(x) == 0
                          or (isinstance(x, str) and x.strip() == "")
            ).sum()),
            "unique": df[col].astype(str).nunique(),   # astype(str) so lists and dicts can be compared
        })
    return pd.DataFrame(rows)


def clean_price(x):
    """Numeric price, plus a flag for 'from X' prices (a minimum, not the real price)."""
    if isinstance(x, str):
        x = x.strip()
        if x in {"—", "-", ""}:
            return pd.Series({"price_clean": np.nan, "price_is_min": False})
        if x.lower().startswith("from"):
            return pd.Series({"price_clean": float(x.split()[-1]), "price_is_min": True})
    value = x if isinstance(x, (int, float)) else np.nan
    return pd.Series({"price_clean": value, "price_is_min": False})

In [ ]:
PALETTE = {
    "violet": {"main": "#7B4FA0", "median": "#4A2C6B", "p95": "#B48FD1"},
    "salmon": {"main": "#F08A72", "median": "#B5503A", "p95": "#F7B5A4"},
}


def plot_distribution(series, title, xlabel, color="violet",
                      clip_quantile=0.995, bins=60, log_x=False, save_path=None):
    """Histogram + KDE with median and 95th percentile reference lines."""
    s = series.dropna()
    colors = PALETTE[color]

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(s, bins=bins, kde=True, stat="density", color=colors["main"],
                 alpha=0.35, edgecolor=None, log_scale=log_x, ax=ax,
                 line_kws={"linewidth": 2})

    median, p95 = s.median(), s.quantile(0.95)
    ax.axvline(median, color=colors["median"], linestyle="--", linewidth=1.5,
               label=f"Median: {median:,.1f}")
    ax.axvline(p95, color=colors["p95"], linestyle=":", linewidth=2,
               label=f"95th percentile: {p95:,.1f}")

    if clip_quantile and not log_x:
        ax.set_xlim(s.min(), s.quantile(clip_quantile))   # hide extreme tail for readability

    ax.set_title(title, fontsize=14, weight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.legend(frameon=False)
    sns.despine()
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

## 📥 Load the data

In [ ]:
data = pd.read_json(DATA_PATH, lines=True)
print(f"{len(data):,} products, {data.shape[1]} columns")

## 1. 🗂️ Dataset structure

**🔍 What we check:** dataset dimensions, column types and nested structures.

`data.info()` gives a first look, but it reports almost everything as `object`. `type_profile` inspects the real type of every value.

In [ ]:
data.info()

In [ ]:
profile = type_profile(data)
profile[["column", "pandas_dtype", "real_types", "n_types", "unique"]]

**💡 Findings**

- 137,269 products and 16 columns.
- Several columns are nested: `categories`, `features`, `description` and `images` are lists, and `details` is a dictionary.
- `price` mixes numbers, `None` and text in the same column 😵 (see section 5).

**🛠️ Decision:** nested columns can't be compared directly as text. They get their own analysis in section 9.

## 2. 🕳️ Data completeness

**🔍 What we check:** null values, empty strings and empty lists or dictionaries.

`isna()` only counts `None` and `NaN`. An empty list (`[]`) or an empty string (`""`) is just as useless, but it doesn't count as null. The gap between `nulls` and `empty` measures these **disguised nulls** 🥸.

In [ ]:
completeness = profile[["column", "nulls", "empty"]].copy()
completeness["missing_pct"] = ((completeness["nulls"] + completeness["empty"]) / len(data) * 100).round(1)
completeness.sort_values("missing_pct", ascending=False)

**💡 Findings**

- Some columns carry almost no information (`bought_together`, `subtitle`, `author`).
- `categories` looks complete according to `isna()`, but 12,637 products have an empty list: a disguised null.
- Some titles are empty, so those products can't be identified at all.

**🛠️ Decision:** columns without information are dropped, and so are products without a title or categories. Fewer fields also means fewer tokens sent to the model 💸.

## 3. 🔑 Uniqueness and duplicates

**🔍 What we check:** whether there is a reliable unique identifier, and how many titles are repeated exactly.

In [ ]:
print("parent_asin is unique:", data["parent_asin"].is_unique)

titles = data["title"].fillna("").str.strip()
repeated = titles[titles != ""].value_counts()
repeated = repeated[repeated > 1]
print(f"{len(repeated):,} titles appear more than once, covering {repeated.sum():,} products")
repeated.head(10)

**💡 Findings**

- `parent_asin` is unique ✅: it's the key used across the whole pipeline (S3 files, vectors and DynamoDB).
- Many titles are repeated exactly. These are the easy duplicates, the ones a plain `duplicated()` would find.

**🛠️ Decision:** exact matching isn't enough. The hard cases are listings of the same product with **different** titles ("GTA: Vice City" vs "Grand Theft Auto: Vice City") and listings with almost **identical** titles that are different products (a disc vs a digital code). That's the gap the pipeline has to cover 🎯.

## 4. 📊 Cardinality and value distribution

**🔍 What we check:** distinct values, category distributions and inconsistent labels.

In [ ]:
data["main_category"].value_counts().head(15)

In [ ]:
leaf_categories = data["categories"].map(lambda c: c[-1] if isinstance(c, list) and c else None)
leaf_categories.value_counts().head(15)

In [ ]:
store = data["store"].dropna()
print(store.nunique(), "→", store.str.lower().str.strip().nunique(), "after normalizing case")
store[store.str.contains("sony", case=False)].value_counts().head(15)

**💡 Findings**

- `main_category` contains values that have nothing to do with video games ("Computers", "Automotive"...) 🤔, because it's the **store department** where the listing was published, not the product type.
- `categories` is a hierarchical path ("Video Games > PlayStation 4 > Games") and describes the product much better.
- The same company is written in many ways in `store` (upper case, truncated names like "SONY COMPUTER ENTERTAINME", resellers).

**🛠️ Decision:** scope and platform come from `categories`. Company names are normalized by the model, not by hand-written rules.

## 5. ✅ Format validity

**🔍 What we check:** data types and formats, starting with the price.

In [ ]:
data[data["price"].map(type) == str]["price"].value_counts().head(10)

In [ ]:
data[["price_clean", "price_is_min"]] = data["price"].apply(clean_price)
print(data["price_is_min"].sum(), "'from' prices")
print(data["price_clean"].isna().sum(), "products without price after cleaning")

**💡 Findings**

- `price` has three ways of saying "no price" 🙃: `None`, a dash (`"—"`) and ranges as text (`"from 19.99"`).
- Converting the column directly with `float()` would fail.

**🛠️ Decision:** `clean_price` stores real prices in a new column (`price_clean`) and flags the "from" prices with `price_is_min`. The original column is kept for traceability.

## 6. 📈 Ranges and outliers

**🔍 What we check:** extreme prices and unexpected rating ranges.

In [ ]:
data[["price_clean", "average_rating", "rating_number"]].describe().round(2)

In [ ]:
plot_distribution(data["price_clean"][data["price_clean"] > 0],
                  "Distribution of product price", "Price (USD, log scale)",
                  color="salmon", log_x=True)

plot_distribution(data["average_rating"],
                  "Distribution of average rating", "Average rating",
                  color="violet", clip_quantile=None, bins=40)

plot_distribution(data["rating_number"],
                  "Distribution of number of ratings", "Number of ratings (log scale)",
                  color="salmon", log_x=True)

**💡 Findings**

- 💰 **Price:** a log-normal distribution centered around $25 (median $24.9; p95 $150.7). There are small bumps at $0.01 and between $1 and $2 (probably DLC or points cards), and a long tail up to about $3,500 (arcade machines and simulators 🕹️).
- ⭐ **Average rating:** high median (4.1). The exact spikes at 1, 2, 3 and 5 are an artifact: products with very few reviews, whose average can only be a whole number.
- 🗳️ **Number of ratings:** median of 21, with a huge spike at a single rating. That explains the spikes in the previous chart.

**🛠️ Decision:** price is not used to detect duplicates, because the same product is sold at very different prices by different sellers. `rating_number` doesn't decide anything either, but it's used to pick the representative listing of each group (the most rated one).

## 7. 🔗 Cross-column consistency

**🔍 What we check:** whether related fields tell the same story.

If `main_category` described the product, it should match the type suggested by the `categories` path. `product_type_from_categories` classifies each product based on that path.

In [ ]:
def product_type_from_categories(cats):
    cats = set(cats) if isinstance(cats, list) else set()
    if "Accessories" in cats or "Controllers" in cats:
        return "accessory"
    if "Consoles" in cats or "Microconsoles" in cats:
        return "console"
    if cats & {"Games", "Digital Games", "Digital Video Games"}:
        return "game"
    return "unknown"


data["product_type_cat"] = data["categories"].apply(product_type_from_categories)
pd.crosstab(data["main_category"], data["product_type_cat"]).sort_values("game", ascending=False).head(10)

In [ ]:
accessory_words = r"\b(?:controller|headset|charger|cable|glove|case|skin|stand|statue|figure)\b"
games = data[data["product_type_cat"] == "game"]
suspicious = games[games["title"].fillna("").str.contains(accessory_words, case=False)]
print(f"{len(suspicious):,} products labelled as games have accessory words in the title")
suspicious["title"].head(10)

**💡 Findings**

- `main_category` and `categories` often disagree, which confirms that `main_category` is the store department.
- Even inside "Games" there are accessories, merchandise and bundles (a boxing glove 🥊, controllers, statues). During extraction, about 14% of products labelled as games turned out not to be games.

**🛠️ Decision:** the category path is a good first filter, but it's not enough. The final product type (game, DLC, bundle, accessory...) is decided by the model during extraction.

## 8. 📝 Text quality

**🔍 What we check:** title length, special tags, abbreviations and variations.

In [ ]:
data["len_title"] = data["title"].str.len()

plot_distribution(data["len_title"], "Distribution of product title length", "Characters", color="violet")
print("Titles over 190 characters:", (data["len_title"] > 190).sum())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for ptype, color in [("game", "#7B4FA0"), ("accessory", "#F08A72")]:
    subset = data.loc[data["product_type_cat"] == ptype, "len_title"]
    sns.kdeplot(subset, fill=True, alpha=0.3, linewidth=2, color=color,
                label=f"{ptype} (n={len(subset):,})", ax=ax)

ax.axvline(200, color="gray", linestyle="--", linewidth=1.2, label="200-char limit")
ax.set_xlim(0, 300)
ax.set_title("Title length by product type", fontsize=14, weight="bold")
ax.set_xlabel("Title length (characters)")
ax.legend(frameon=False)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
title_stats = (data.assign(over_200=data["len_title"] > 200)
                   .groupby("main_category")
                   .agg(mean=("len_title", "mean"), median=("len_title", "median"),
                        max=("len_title", "max"), over_200=("over_200", "sum")))
title_stats.sort_values("max", ascending=False).round(1).head(10)

In [ ]:
tags = data["title"].fillna("").str.findall(r"\[([^\]]+)\]").explode().dropna()
tags.value_counts().head(15)

In [ ]:
for term in ["vice city", "black ops ii", "black ops 2"]:
    print(f"\n'{term}':")
    print(data.loc[data["title"].fillna("").str.lower().str.contains(term), "title"].head(4).to_string())

**💡 Findings**

- Title length has **two populations**: a peak between 25 and 40 characters (clean titles like "FIFA 23 - PlayStation 4") and another one right before 200 characters. Median 66, p95 193.
- That second peak is mostly **accessories** 🎧, whose sellers stuff the title with keywords up to Amazon's limit (9,025 titles are longer than 190 characters).
- **Tags in brackets** carry real attributes: `[Digital Code]` and `[Download]` mean digital format, `[Japan Import]` means region 🇯🇵.
- The same game is written in several ways: abbreviations ("GTA"), Roman numerals ("Black Ops II" vs "Black Ops 2") and extra words ("Platinum", "Renewed").

**🛠️ Decision:** comparing titles directly would penalize the similarity between a short title and a keyword-stuffed one. That's why the model first extracts a normalized `base_title`, and these patterns become explicit rules and examples in the extraction prompt 🧠.

## 9. 🪆 Nested-field analysis

**🔍 What we check:** which keys live inside `details`, whether `author` applies to video games, and what the `categories` hierarchy looks like.

The coverage calculation shows the percentage of products where each dictionary key appears.

In [ ]:
keys = Counter(k for d in data["details"] if isinstance(d, dict) for k in d)
print(len(keys), "distinct keys in details")

coverage = pd.Series(keys).div(len(data)).mul(100).round(1).sort_values(ascending=False)
coverage.head(20)

In [ ]:
pricing = data["details"].map(lambda d: d.get("Pricing") if isinstance(d, dict) else None).dropna()
print("Products with details.Pricing:", len(pricing))
print("Distinct values:", pricing.nunique())
pricing.value_counts().head(3)

In [ ]:
author_keys = Counter(k for d in data["author"] if isinstance(d, dict) for k in d)
print(len(author_keys), "distinct keys in author")
pd.Series(author_keys).div(len(data)).mul(100).round(2).sort_values(ascending=False)

In [ ]:
depth = data["categories"].map(lambda c: len(c) if isinstance(c, list) else 0)
depth.value_counts().sort_index()

**💡 Findings**

`details` has more than a hundred different keys, but only a few are worth keeping:

- 🏷️ **Useful to identify the product:** `Manufacturer` (84.1%, needs normalizing like `store`) and `Item model number` (43.2%, the strongest identifier when both listings have it).
- 🧭 **Useful to classify the type:** `Type of item` (55%) and `Rated` (30.2%, the age rating, which only games have).
- 🪤 **Misleading:** `Pricing` (61.1%) is not a price but the same legal notice repeated; `Date First Available` (93.6%) is the date the listing was created, not the release date. High coverage doesn't mean useful information!

Also, `author` appears in only 0.2% of products and describes book or content authors: it doesn't apply to video games. `categories` paths have different depths, and the platform usually shows up in them.

**🛠️ Decision:** only four `details` keys are sent to the model (`Manufacturer`, `Type of item`, `Rated`, `Item model number`), together with the title, the store, the categories and the first three features. Misleading fields and `author` are dropped.

## 10. 🏁 Scope definition: final dataset

**🔍 What we check:** the size of the dataset when filtered to the 20 best-selling video game franchises.

The list comes from Wikipedia's [List of best-selling video game franchises](https://en.wikipedia.org/wiki/List_of_best-selling_video_game_franchises). Using an external source avoids cherry-picking examples 🍒, and these are the products with the most traffic, where a duplicate hurts the most.

The patterns use word boundaries (`\b`) to avoid false matches, like "gta" inside another word.

In [ ]:
TOP20_FRANCHISES = {
    "Mario": r"\bmario\b",
    "Tetris": r"\btetris\b",
    "Pokémon": r"\bpok[eé]mon\b",
    "Call of Duty": r"\bcall of duty\b",
    "Grand Theft Auto": r"\bgrand theft auto\b|\bgta\b",
    "Minecraft": r"\bminecraft\b",
    "FIFA": r"\bfifa\b",
    "Assassin's Creed": r"\bassassin'?s creed\b",
    "Resident Evil": r"\bresident evil\b",
    "Final Fantasy": r"\bfinal fantasy\b",
    "Wii (series)": r"\bwii (?:sports|fit|play|party|music)\b",
    "Lego": r"\blego\b",
    "The Sims": r"\bthe sims\b",
    "NBA 2K": r"\bnba 2k",
    "The Legend of Zelda": r"\bzelda\b",
    "Madden NFL": r"\bmadden\b",
    "Need for Speed": r"\bneed for speed\b",
    "Sonic the Hedgehog": r"\bsonic\b",
    "Monster Hunter": r"\bmonster hunter\b",
    "Red Dead": r"\bred dead\b",
}

In [ ]:
has_title = data["title"].fillna("").str.strip() != ""
has_categories = data["categories"].map(lambda c: isinstance(c, list) and len(c) > 0)
clean = data[has_title & has_categories]
games = clean[clean["categories"].map(lambda c: "Games" in c)]

game_titles = games["title"].str.lower()
per_franchise = pd.Series({name: game_titles.str.contains(p).sum()
                           for name, p in TOP20_FRANCHISES.items()})

combined = "|".join(f"(?:{p})" for p in TOP20_FRANCHISES.values())
selected = games[game_titles.str.contains(combined)]

print(f"{len(selected):,} unique products in the top-20 franchises")
per_franchise.sort_values(ascending=False)

In [ ]:
funnel = pd.DataFrame({
    "stage": ["Raw dataset", "With title and categories", "Category 'Games'", "Top-20 franchises"],
    "products": [len(data), len(clean), len(games), len(selected)],
})
funnel["% of raw"] = (funnel["products"] / len(data) * 100).round(1)
funnel

**💡 Findings**

- All 20 franchises are present in the catalog 🎉.
- The final scope has **3,740 products**: big enough to be representative, and small enough to iterate on the prompt quickly and cheaply.
- A product can belong to more than one franchise ("Mario & Sonic" 🍄🦔), which is why the per-franchise sum is higher than the total.

**🛠️ Decision:** these 3,740 products are the input of the pipeline. The same filter is applied in the `game-catalog-prepare-batch` Lambda through the `title_contains` parameter, and its result matches this notebook.

## 🧾 Summary: from EDA findings to pipeline decisions

| 💡 Finding | 🛠️ Pipeline decision |
|---|---|
| `parent_asin` is unique | Key for S3, vectors and DynamoDB |
| Disguised nulls (empty lists and strings) | Removed in the ETL, before calling the model |
| Columns without information (`author`, `bought_together`...) | Dropped: fewer tokens, lower cost |
| `main_category` is the store department | Scope and platform taken from `categories` |
| Accessories and merchandise labelled as games | Product type decided by the model |
| Tags, abbreviations and Roman numerals in titles | Explicit rules and examples in the prompt |
| `details.Pricing` and `Date First Available` are misleading | Not sent to the model |
| Exact title matching misses the hard cases | Extraction + embeddings + decision rules |
| Top-20 franchises: 3,740 products | MVP scope |

Ready for the pipeline! 🚀